# 🕵️‍♂️ The Quantum Heist: Breaking RSA & The PQC Shield
**Event:** QCoffee Hangout (Tripoli University)  
**Stack:** Qiskit 1.x / Python 3.12

---

### 📜 Mission Briefing
You are part of a "Red Team" (Ethical Hackers) in the year 2030. 
We have intercepted a secure transmission between **Alice Corp** and **Bob Bank**.

**Your Objective:**
1. Analyze the intercepted **Public Key** (The Lock).
2. Use a **Quantum Computer simulation** to break the key (Shor's Algorithm).
3. Recover the **Private Key** and decrypt the secret message.
4. Test if this attack works against **Post-Quantum Cryptography (PQC)**.

In [ ]:
# @title 🛠️ Step 0: Mission Setup (Run this first)
# Installing dependencies compatible with Python 3.12 and Qiskit 1.x
!pip install qiskit qiskit-aer qiskit-algorithms cryptography pylatexenc matplotlib

import qiskit
import qiskit_algorithms
from qiskit_algorithms import Shor
from qiskit_aer.primitives import Sampler # Critical for Qiskit 1.x
from qiskit.visualization import plot_histogram
import numpy as np
import time

print("✅ System Ready.")
print(f"🔹 Qiskit Version: {qiskit.__version__}")
print(f"🔹 Algorithms Version: {qiskit_algorithms.__version__}")

## 📡 Module 1: The Interception (Understanding TLS/SSL)

When you visit a website (like your bank), your browser uses **TLS**. It relies on a **Public Key** ($N$) and a **Private Key**.

Mathematically: $N = p \times q$. 
If we find $p$ and $q$, we break the lock.

In [ ]:
# --- SIMULATION: THE INTERCEPTION ---

def get_target_key():
    # We use N=15 for simulation speed.
    # Real RSA keys (2048 bits) require millions of qubits.
    N = 15  
    e = 7   
    return N, e

N, e = get_target_key()

print(f"📡 INTERCEPTED PACKET HEADER")
print(f"---------------------------")
print(f"Protocol: TLS 1.3 (Simulated)")
print(f"Server Identity: BANK_OF_TRIPOLI_SECURE")
print(f"⚠️ PUBLIC KEY FOUND (N): {N}")
print(f"Encryption Exponent (e): {e}")
print(f"\nTarget Analysis: To break this lock, we must find the prime factors of {N}.")

## 🔒 Module 2: The Locked Message

We also captured the message payload. 
$$ C = M^e \pmod N $$

In [ ]:
# --- SIMULATION: THE ENCRYPTED PAYLOAD ---

# Our secret flag is represented by the number '2'
secret_message_int = 2  

# Encrypting it using the Public Key
encrypted_msg = (secret_message_int ** e) % N

print(f"📦 CAPTURED PAYLOAD")
print(f"-------------------")
print(f"Encrypted Data (Ciphertext): {encrypted_msg}")
print(f"To decrypt, we need the Private Key (d).")

## ⚛️ Module 3: The Quantum Attack (Shor's Algorithm)

We use **Shor's Algorithm** via Qiskit 1.x Primitives.

1. **Superposition:** Guess all factors at once.
2. **Interference:** Cancel out wrong answers.

**Note:** We use the `Sampler` primitive which is the modern standard for running algorithms on Aer.

In [ ]:
# --- ACTION: RUN QUANTUM SIMULATION ---
print(f"🚀 INITIATING QUANTUM ATTACK ON N={N}...")

# 1. Setup the Sampler (Modern Qiskit 1.x Primitive)
# This replaces the old 'backend.run' or 'QuantumInstance' methods
sampler = Sampler()

# 2. Setup Shor's Algorithm with the Sampler
shor = Shor(sampler=sampler)

# 3. EXECUTE
start_time = time.time()
try:
    result = shor.factor(N)
    success = True
except Exception as e:
    print(f"Simulation Error: {e}")
    success = False

end_time = time.time()

# 4. Display Results
if success:
    print(f"\n✅ QUANTUM PROCESS COMPLETE")
    print(f"---------------------------")
    print(f"Time taken: {round(end_time - start_time, 2)} seconds")
    print(f"Factors Found: {result.factors}")

    if result.factors:
        p_found = result.factors[0][0]
        q_found = result.factors[0][1]
        print(f"Calculated Primes: p={p_found}, q={q_found}")
    else:
        print("Simulation result ambiguous. Try running again.")

## 🔓 Module 4: The Breach (Decryption)

Reconstructing the Private Key ($d$) using the factors found by the Quantum Computer.

In [ ]:
# --- ACTION: REGENERATE PRIVATE KEY ---

if 'p_found' in locals():
    # 1. Calculate Phi
    phi = (p_found - 1) * (q_found - 1)

    # 2. Calculate Private Key (d)
    d = pow(e, -1, phi)

    print(f"🗝️ PRIVATE KEY GENERATED")
    print(f"-----------------------")
    print(f"Private Key (d): {d}")

    # 3. Decrypt
    decrypted_msg = (encrypted_msg ** d) % N

    print(f"\n🔓 DECRYPTING PAYLOAD...")
    print(f"Decrypted Integer: {decrypted_msg}")

    if decrypted_msg == secret_message_int:
        print("\n🎉 SUCCESS! MESSAGE INTERCEPTED.")
        print("Flag Content: 'MEETING AT SARAYA HALL @ 5PM'")
    else:
        print("Decryption failed.")
else:
    print("⚠️ Cannot decrypt: Quantum factors not found.")

## 🛡️ Module 5: The Shield (PQC Migration)

**Post-Quantum Cryptography (PQC)** uses Lattice-based math, which Shor's algorithm cannot solve.

In [ ]:
# --- SIMULATION: ATTACKING PQC ---

print(f"⚔️ ATTEMPTING ATTACK ON KYBER-512 KEY...")

try:
    lattice_key = "Vector[512]" 
    
    if isinstance(lattice_key, int):
        shor.factor(lattice_key)
    else:
        raise ValueError("❌ ERROR: Input is not an integer. Shor's Algorithm cannot process Lattices.")

except Exception as error:
    print(error)
    print("\n🛡️ CONCLUSION: The Quantum Attack failed.")
    print("This demonstrates the resilience of PQC.")